# <mark> 3번 : 동일한 데이터셋에서 ResNet과 VGG16을 각각 학습시켜 성능을 비교하세요. </mark>

In [ ]:
# ---------------------------------------------------------
# 런타임 유형을 변경하고, 잘 적용되었는지 확인용입니다.
# ---------------------------------------------------------
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

Thu Jun 11 14:57:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 위클리챌린지 1번, 2번에 이어, 똑같은 데이터 사용
# ---------------------------------------------------------

!curl -L "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz" -o flowers.tgz
!tar -xzf flowers.tgz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  218M  100  218M    0     0  16.1M      0  0:00:13  0:00:13 --:--:-- 17.1M


In [ ]:
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchvision import models

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 데이터 전처리 -> 로더 연결 -> GPU로 올리기
# ---------------------------------------------------------

transform = transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor(),])
full_dataset = ImageFolder(root='./flower_photos', transform=transform)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = [(inputs.to(device), labels.to(device)) for inputs, labels in train_loader]
test_loader = [(inputs.to(device), labels.to(device)) for inputs, labels in test_loader]

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 데이터셋을 확인하고 자동으로 출력 클래스의 개수를 num_classes로 선언.
# ---------------------------------------------------------

print(full_dataset.classes, '\n')

num_classes = len(full_dataset.classes)
print(f"자동으로 확인된 클래스 개수: {num_classes}")
print(f"학습 데이터 개수: {train_size}")

['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips'] 

자동으로 확인된 클래스 개수: 5
학습 데이터 개수: 2936


In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 1) ResNet50 선언 후 데이터 학습 및 테스트
# 이때, 바닥부터 학습시키는 걸로 동일하게 진행한다.
# ---------------------------------------------------------

base_model_resnet = models.resnet50(weights=None)
base_model_resnet = nn.Sequential(*list(base_model_resnet.children()))[:-2]

In [ ]:
class CustomResNet50(nn.Module):
  def __init__(self, num_classes):
    super(CustomResNet50, self).__init__()
    self.base_model_resnet = base_model_resnet
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
    self.fc1 = nn.Linear(2048, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)
    self.softmax = nn.Softmax(dim=1)

  def forward(self, x):
    x = self.base_model_resnet(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x,1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    # x = self.softmax(x) --- 🤖 파이토치 CrossEntropyLoss는 이미 Softmax를 가지고 있다
    return x

model_resnet = CustomResNet50(num_classes = num_classes)

model_resnet = model_resnet.to(device)

In [ ]:
criterion_resnet = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(model_resnet.parameters(), lr=0.0001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
  model_resnet.train()
  running_loss = 0.0
  for i, l in train_loader:
    optimizer_resnet.zero_grad()
    outputs = model_resnet(i)
    loss = criterion_resnet(outputs, l)
    loss.backward()
    optimizer_resnet.step()
    running_loss += loss.item()
  print(f'epoch: {epoch+1}, Loss: {running_loss/len(train_loader)}')

epoch: 1, Loss: 1.319876975339392
epoch: 2, Loss: 1.050629256212193
epoch: 3, Loss: 0.9621941609227139
epoch: 4, Loss: 0.8809715140125026
epoch: 5, Loss: 0.8000070251848387
epoch: 6, Loss: 0.6823804061050001
epoch: 7, Loss: 0.6348668276939703
epoch: 8, Loss: 0.5890434253151002
epoch: 9, Loss: 0.5464252803636633
epoch: 10, Loss: 0.41187272237046907


In [ ]:
model_resnet.eval()
correct = 0
total = 0

with torch.no_grad():
  for i, l in test_loader:
    outputs = model_resnet(i)
    _, predicted = torch.max(outputs, 1)
    total += l.size(0)
    correct += (predicted == l).sum().item()

accuracy_resnet = correct/total
print(f'test accuracy : {accuracy_resnet * 100}%')

test accuracy : 62.94277929155313%


In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 2) VGG16 선언 후 데이터 학습 및 테스트
# 이때, ResNet과 마찬가지로 바닥부터 학습시키는 걸로 동일하게 진행한다.
# ---------------------------------------------------------

base_model_vgg = models.vgg16(weights=None)

In [ ]:
class CustomVGG16(nn.Module):
  def __init__(self, num_classes):
    super(CustomVGG16, self).__init__()
    self.base_model_vgg = base_model_vgg
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
    self.fc1 = nn.Linear(512, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)
    self.softmax = nn.Softmax(dim=1)

  def forward(self, x):
    x = self.base_model_vgg.features(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x,1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    # x = self.softmax(x) --- 🤖 파이토치 CrossEntropyLoss는 이미 Softmax를 가지고 있다
    return x

model_vgg = CustomVGG16(num_classes = num_classes)
model_vgg = model_vgg.to(device)

In [ ]:
criterion_vgg = nn.CrossEntropyLoss()
optimizer_vgg = optim.Adam(model_vgg.parameters(), lr=0.0001)

In [ ]:
for epoch in range(num_epochs): #resnet 과 마찬가지로 10번 학습 진행
  model_vgg.train()
  running_loss = 0.0
  for i, l in train_loader:
    optimizer_vgg.zero_grad()
    outputs = model_vgg(i)
    loss = criterion_vgg(outputs, l)
    loss.backward()
    optimizer_vgg.step()
    running_loss += loss.item()
  print(f'epoch: {epoch+1}, Loss: {running_loss/len(train_loader)}')

epoch: 1, Loss: 1.4094878292602042
epoch: 2, Loss: 1.254607634699863
epoch: 3, Loss: 1.1386091618434242
epoch: 4, Loss: 1.0430542956227842
epoch: 5, Loss: 0.9486353105824926
epoch: 6, Loss: 0.911000313318294
epoch: 7, Loss: 0.8698284638964612
epoch: 8, Loss: 0.7837054975654768
epoch: 9, Loss: 0.7829769010777059
epoch: 10, Loss: 0.7388306655313658


In [ ]:
model_vgg.eval()
correct = 0
total = 0

with torch.no_grad():
  for i, l in test_loader:
    outputs = model_vgg(i)
    _, predicted = torch.max(outputs, 1)
    total += l.size(0)
    correct += (predicted == l).sum().item()

accuracy_vgg = correct/total
print(f'test accuracy : {accuracy_vgg * 100}%')

test accuracy : 68.11989100817438%


In [ ]:
print("동일한 데이터셋에서 ResNet과 VGG16을 각각 학습시켜 성능을 비교하였을 때,")
print(f'test accuracy : {accuracy_resnet * 100}%')
print(f'test accuracy : {accuracy_vgg * 100}%')

동일한 데이터셋에서 ResNet과 VGG16을 각각 학습시켜 성능을 비교하였을 때,
test accuracy : 62.94277929155313%
test accuracy : 68.11989100817438%


In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 첫 번째 실행하였을 때 (둘 다 softmax를 지우기 전) 결과:
# test accuracy (resnet) : 67.16621253405994%
# test accuracy (vgg16) : 24.250681198910083%
# 10번 학습으로 이 vgg 보다 resnet이 성능이 더 좋다고 판단할 수 없다고 생각한다.
# 또한, 데이터를 shuffle 하기 때문에 다시 실행할 경우 각각의 accuracy가 또 달라질거라 생각한다.
# ---------------------------------------------------------

In [ ]:
# ----------------------- 🤖 제미나이 -----------------------
# 두 번째 실행하였을 때 (둘 다 softmax를 지우고 나서) 결과:
# test accuracy (resnet) : 62.94277929155313%
# test accuracy (vgg16) : 68.11989100817438%
#
# 분석결과:
# ResNet50:
# 층이 50개나 되고 파라미터 수가 매우 많아 표현력이 강력합니다. 하지만 데이터 개수가 적은 상태에서
# 10에폭 동안 바닥부터 학습(weights=None)시키면, 모델이 데이터의 핵심 특징을 찾기보다 학습 데이터 자체를
# 통째로 외워버리는 과적합(Overfitting)에 빠지기 쉽습니다. 그래서 오히려 시험 데이터(Test Accuracy)에서는
# 점수가 낮게 나올 수 있습니다.
# VGG16: 구조가 상대적으로 단순하고 직관적입니다. 데이터 수가 적을 때는 너무 복잡한 모델보다 VGG처럼
# 적당히 두터운 모델이 과적합을 피해 가며 더 안정적인 일반화 성능을 내기도 합니다.
# ---------------------------------------------------------

# <font color='red'>1. 배운 점</font>


<font color='green'>1.   클래스 개수의 자동 감지
- `num_classes = 5`처럼 직접 숫자를 적는 하드코딩 대신, `num_classes = len(full_dataset.classes)`로 자동 할당하는 방식을 학습. ImageFolder는 "메인 폴더 아래 클래스별 하위 폴더" 구조를 인식해 폴더 개수와 이름을 `.classes` 속성에 담아두므로, 데이터셋이 바뀌어도 코드를 수정할 필요가 없는 안전한 패턴임을 확인.
2.   스킵 커넥션은 어디에 숨어 있는가 — 캡슐화의 이해
- 커스텀 모델의 forward 함수만 보면 스킵 커넥션이 전혀 안 보여서 의문이 들었는데, ResNet50의 핵심인 x+F(x) 연산은 `base_model` 내부의 Bottleneck 블록들 안에 통째로 캡슐화되어 있음을 확인. `x = self.base_model_resnet(x)` 한 줄이 실행되는 순간 내부에서 수십 개의 스킵 커넥션 연산이 일어나며, `print(resnet)`으로 Bottleneck 구조와 downsample을 직접 눈으로 확인 가능.
3.   Softmax + CrossEntropyLoss 중복이 모델을 죽이는 실제 현장 목격
- `nn.CrossEntropyLoss`는 내부에 LogSoftmax + NLLLoss를 포함하므로 날것의 점수(로짓)를 기대하는데, forward 끝에서 Softmax를 한 번 더 적용하면 출력이 0~1로 압축된 상태에 추가 연산이 겹쳐 그래디언트가 0으로 소실됨.
- 실제로 VGG16은 3에포크부터 Loss가 1.6597…에 소수점 끝자리까지 동결되어 정확도 24.25%(5클래스 찍기 확률 20% 수준)라는 식물인간 상태가 됨을 목격.
4.   동일한 악조건에서 구조가 만든 생존 격차
- 같은 Softmax 중복 조건에서도 ResNet50은 스킵 커넥션 덕분에 기울기가 깊은 층까지 전달되어 67%까지 학습된 반면, 스킵 커넥션이 없는 단순 적층 구조의 VGG16은 기울기 소실로 완전히 멈춤. 구조적 차이가 학습 안정성에 미치는 영향을 두 모델의 대조 실험으로 직접 확인.

# <font color='red'>2. 어려웠던 점</font>


<font color='green'>1.   연쇄 오타와 에러 메시지 해독 훈련
- 커스텀 모델 작성 과정에서 직접 낸 오타들로 인한 에러를 연달아 디버깅.
  - `fasward` 오타 → NotImplementedError(파이토치는 forward라는 이름의 메서드를 자동으로 찾음)
  - `x = self.base_model_resnet`에 (x) 누락 → AttributeError(데이터 대신 Sequential 객체 자체가 x에 대입됨)
  -  `global_avg_pool(x, 1)` → TypeError(AdaptiveAvgPool2d는 입력 텐서 하나만 받음)
  - `loss.backwork` → backward 오타.
2.   VGG 전체를 통과시켰을 때의 차원 붕괴
- ResNet과 똑같이 `self.base_model_vgg(x)`로 전체를 통과시키자 ValueError: Input dimension should be at least 3 발생.
- VGG16은 features → avgpool → classifier가 한 덩어리라 전체를 통과하면 이미 (배치, 1000)의 납작한 2차원 벡터가 되어 나오므로, 풀링이 필요로 하는 공간 정보(H, W)가 살아있는 4차원 텐서가 아니게 됨.
- `self.base_model_vgg.features(x)`로 특징 추출부만 통과시켜 (배치, 512, 7, 7) 상태를 유지해야 함을 이해.
3.   "왜 24%인가"의 진단
- VGG의 처참한 점수가 모델 성능 문제인지 코드 문제인지 판별하는 과정이 어려웠음.
- AI의 도움으로 Loss가 소수점 끝자리까지 동결된 패턴(업데이트 자체가 멈춘 신호)을 단서로 Softmax 중복이라는 진범을 찾아냈고, 지난 과제의 보완점이 실제로 모델을 죽일 수 있는 치명적 버그임을 체감.

# <font color='red'>3. 앞으로의 보완점</font>


<font color='green'>1.   고정 시드 기반의 공정 비교 환경 구축
- 현재 random_split과 shuffle에 시드가 고정되어 있지 않아 실행할 때마다 학습/테스트 구성이 달라지므로, 재현 가능한 비교를 위해 시드를 고정하고 두 모델이 완전히 동일한 분할을 사용하도록 보완.
2.   에포크 확대 및 학습 곡선 비교
- 10에포크는 바닥 학습이 수렴하기에 매우 짧고 모델별 수렴 속도가 다르므로, 에포크를 늘려 학습 곡선 전체를 비교해야 공정한 우열 판단이 가능. 5% 내외의 점수 차이로 결론 내리지 않기.
3.   전이 학습 조건에서의 재대결
- 두 모델 모두 weights=DEFAULT로 사전 학습 가중치를 불러온 상태에서 동일 비교를 진행하여, 바닥 학습이라는 핸디캡을 제거한 진검승부 결과를 확인.

# <font color='red'>4. 수행 과정 종합 평가</font>


<font color='green'>1.   서론
- 본 과제는 동일한 꽃 이미지 데이터셋(flower_photos, 5개 클래스)에서 ResNet50과 VGG16을 각각 바닥부터(weights=None) 학습시켜 성능을 비교하는 것을 목표로 진행되었습니다. 공정한 비교를 위해 두 모델 모두 동일한 커스텀 헤드(글로벌 평균 풀링 → fc1 → ReLU → fc2)를 부착하고, 동일한 데이터 분할·배치 크기·옵티마이저(Adam, lr=0.0001)·에포크(10회) 조건을 적용했으며, Google Colab GPU 환경에서 수행했습니다.

<font color='green'>2.   본론


학습은 디버깅과 두 차례의 비교 실험으로 전개되었습니다.

- 디버깅 단계:
커스텀 ResNet50 작성 중 발생한 일련의 오타(fasward, 레이어 통과 괄호 누락, 풀링 인자 오류, backwork)를 에러 메시지를 단서로 차례로 수정했습니다. 이어 VGG16에서는 모델 전체를 통과시켜 발생한 차원 붕괴(ValueError)를 만나, VGG는 features만 분리해 사용해야 4차원 텐서가 유지된다는 구조적 차이를 파악하고 해결했습니다.
- 1차 비교 실험 (Softmax 포함 상태):
  *   ResNet50은 67.17%를 기록했으나 VGG16은 24.25%로 사실상 무작위 추측 수준에 그쳤습니다. VGG의 Loss가 3에포크부터 1.6597…에 소수점 끝자리까지 동결된 패턴을 단서로, 출력층 Softmax와 CrossEntropyLoss 내부 연산의 중복이 기울기 소실을 일으킨 원인임을 진단했습니다. 같은 조건에서 ResNet만 학습이 진행된 것은 스킵 커넥션이 기울기를 깊은 층까지 전달해 준 덕분이었습니다.
- 2차 비교 실험 (Softmax 제거 후):
  *   두 모델 모두 forward에서 Softmax를 제거하고 로짓을 직접 손실 함수에 전달하도록 수정한 결과, ResNet50 62.94% 대 VGG16 68.12%로 VGG가 24.25%에서 68.12%로 극적으로 회복하며 오히려 역전했습니다.

<font color='green'>3.   결론
- 최종 결과만 보면 VGG16이 약 5.2%p 앞섰지만, 학습 데이터 약 2,900장의 작은 데이터셋에서 파라미터가 많은 ResNet50은 과적합에 빠지기 쉽고, 10에포크는 수렴에 크게 못 미치는 짧은 시간이며, 시드 미고정으로 실행마다 데이터 구성이 달라지므로 이 결과로 두 모델의 우열을 단정할 수 없다고 판단했습니다. 실험 전 스스로 메모했던 "10번 학습으로 우열을 판단할 수 없고, 셔플 때문에 재실행 시 정확도가 달라질 것"이라는 예측이 분석을 통해 검증되었습니다.
- 무엇보다 손실 함수가 기대하는 입력(로짓)과 모델 출력의 규약을 어기는 것만으로 거대한 모델이 완전히 학습 불능에 빠질 수 있다는 것, 그리고 그 동일한 악조건을 스킵 커넥션의 유무가 생존과 사망으로 갈라놓는다는 것을 두 모델의 대조를 통해 직접 확인한 실험이었습니다.

🤖 제미나이 대화 원본 : https://gemini.google.com/share/bf01fca5c6fd


수행시간 : **1시반 10분 소요**